# Project 6 -- Calgary Neighborhood Livability Segmentation
## Exploratory Data Analysis

This notebook loads all four source datasets from the Calgary Open Data portal,
engineers community-level features, and performs initial exploratory analysis
including distributions, correlations, and a first-pass clustering exploration.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Add src/ to path so we can import project modules
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import (
    load_all_datasets,
    build_feature_matrix,
    FEATURE_COLUMNS,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
print("Imports OK")

## 1. Load Raw Datasets

We fetch (or load from cache) all four datasets and inspect their shapes and
first few rows.

In [ ]:
datasets = load_all_datasets(force_refresh=False)

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"Dataset: {name}  |  Shape: {df.shape}")
    print(f"{'=' * 60}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(3))

## 2. Build Community Feature Matrix

Aggregate each dataset to the community level and merge into a single
feature matrix with 10 features per community.

In [ ]:
raw_df, scaled_df, scaler = build_feature_matrix(datasets=datasets)

print(f"Feature matrix shape: {raw_df.shape}")
print(f"Feature columns: {FEATURE_COLUMNS}")
raw_df.head(10)

In [ ]:
raw_df.describe().round(2)

## 3. Feature Distributions

Histograms for each of the 10 community-level features.

In [ ]:
for col in FEATURE_COLUMNS:
    fig = px.histogram(
        raw_df,
        x=col,
        nbins=30,
        title=f"Distribution of {col.replace('_', ' ').title()}",
        marginal="box",
    )
    fig.update_layout(height=350, width=700)
    fig.show()

## 4. Correlation Heatmap

Examine pairwise Pearson correlations among the 10 features.

In [ ]:
corr = raw_df[FEATURE_COLUMNS].corr().round(2)

fig_corr = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Feature Correlation Heatmap",
    labels={"color": "Pearson r"},
)
fig_corr.update_layout(height=600, width=700)
fig_corr.show()

## 5. Initial Clustering Exploration

### 5.1 Elbow Curve and Silhouette Scores

In [ ]:
X = scaled_df[FEATURE_COLUMNS].values

k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

elbow_df = pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": silhouettes})

fig_elbow = px.line(elbow_df, x="k", y="inertia", markers=True, title="Elbow Curve")
fig_elbow.show()

fig_sil = px.line(elbow_df, x="k", y="silhouette", markers=True, title="Silhouette Score vs k")
fig_sil.show()

optimal_k = elbow_df.loc[elbow_df["silhouette"].idxmax(), "k"]
print(f"Optimal k (best silhouette): {optimal_k}")

### 5.2 KMeans with Optimal k

In [ ]:
km_final = KMeans(n_clusters=int(optimal_k), random_state=42, n_init=10)
raw_df["cluster"] = km_final.fit_predict(X)

print(f"Cluster sizes:\n{raw_df['cluster'].value_counts().sort_index()}")

# Profile
profile = raw_df.groupby("cluster")[FEATURE_COLUMNS].mean().round(2)
display(profile)

### 5.3 PCA Scatter Plot

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["community"] = raw_df["community"].values
pca_df["cluster"] = raw_df["cluster"].astype(str).values

fig_pca = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="cluster",
    hover_name="community",
    title="PCA -- Communities coloured by KMeans Cluster",
    width=800,
    height=550,
)
fig_pca.show()

print(f"\nExplained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.2%}, "
      f"Total={sum(pca.explained_variance_ratio_):.2%}")

## 6. Summary

Key observations from EDA:

- The feature matrix contains 10 community-level features derived from four
  separate City of Calgary datasets.
- Several features (total population, business count, total crimes) are
  right-skewed, reflecting the concentration of activity in a few large
  communities.
- Crime rate and business diversity tend to correlate, suggesting that
  economically active areas also report more crime.
- The elbow curve and silhouette analysis point towards a small number of
  distinct livability segments.
- PCA confirms that 2 components capture a meaningful share of variance and
  separate the clusters visually.

Next steps: refine clustering parameters, explore hierarchical clustering,
and build the interactive Streamlit dashboard.